# PT/PB vs PT/NALM — B target comparison

Both cell systems share the same **patient T cells**; the **B target** differs:

* **PT/PB**   = `patient B + patient T`  — autologous primary patient B-ALL
* **PT/NALM** = `NALM-6 + patient T`     — B-ALL cell-line target

Because the T compartment is held fixed, the analyses below ask:

1. **T-cell side (CD8):** does the patient T cell respond differently to autologous primary B-ALL vs the NALM-6 line?
2. **B-cell side:** how does primary patient B-ALL surface biology compare to the NALM-6 model under the same T donor?

Sample availability:

| Time × Condition | PT/PB | PT/NALM |
| --- | --- | --- |
| 6h Mock          | S009 | S013 |
| 6h Blinatumomab  | S010 | S014 |
| 48h Mock         | S011 | — (missing, was S015) |
| 48h Blinatumomab | S012 | S016 |

PT/NALM has no 48h Mock, so cross-system analyses that need both Mock & Blina are run only at **6h**.


In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

print(f'scvi-tools: {scvi.__version__}')

from nalm_utils import *

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'

# Two systems being compared (both use patient T donor)
SYS_PT_PB   = 'patient B + patient T'    # PT/PB    — autologous primary B-ALL
SYS_PT_NALM = 'NALM-6 + patient T'       # PT/NALM  — NALM-6 cell-line target


In [ ]:
# [1 · Data loading]
adata = sc.read_h5ad(ANNOTATED_CACHE)

# Sample availability across the two systems being compared
mask_sys = adata.obs['cell_system'].isin([SYS_PT_PB, SYS_PT_NALM])
print(f'Total cells in PT/PB + PT/NALM: {mask_sys.sum()}')
pd.crosstab(
    index=[adata.obs[mask_sys]['cell_system'], adata.obs[mask_sys]['sample']],
    columns=[adata.obs[mask_sys]['time'], adata.obs[mask_sys]['condition']],
)


In [ ]:
# [1b · UMAP exploration]
mask_explore = (
    (adata.obs['cell_type_annot'].isin(['CD4', 'CD8', 'B'])) &
    (adata.obs['cell_system'].isin([SYS_PT_PB, SYS_PT_NALM]))
)
adata_sub = adata[mask_explore].copy()

print(f'Cells in PT/PB + PT/NALM (CD4/CD8/B): {adata_sub.n_obs}')
pd.crosstab(index=adata_sub.obs['sample'],
            columns=[adata_sub.obs['cell_system'], adata_sub.obs['cell_type_annot']])

sc.pl.umap(adata_sub,
  color=['CD3e', 'CD19', 'CD8', 'cell_system', 'cell_type_annot', 'condition'],
  layer='arcsinh', frameon=False)


## Cross-condition comparison — CD8 across time × condition


In [ ]:
# [2 · 4-way DA panel: CD8 — PT/PB vs PT/NALM across time × condition]
# (PT/NALM has no 48h Mock, so that group is missing on the comparison side.)
# B cell markers are dropped to keep the focus on T-cell biology
# (potential B-cell contamination from either patient B or NALM-6).
def _load_b_panel():
    for name in ('b_cell_markers', 'or'):
        try:
            return load_marker_panel(name)
        except KeyError:
            continue
    raise KeyError('No B cell marker panel found in marker_panels.json')

B_CELL_MARKERS_SET = set(_load_b_panel())

mask_pb_cd8 = (
    (adata.obs['cell_system'] == SYS_PT_PB) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_pb_cd8 = adata[mask_pb_cd8].copy()
adata_pb_cd8.obs['time_cond'] = (
    adata_pb_cd8.obs['time'].astype(str) + ' ' + adata_pb_cd8.obs['condition'].astype(str)
)
adata_pb_cd8 = adata_pb_cd8[:, ~adata_pb_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

mask_nm_cd8 = (
    (adata.obs['cell_system'] == SYS_PT_NALM) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_nm_cd8 = adata[mask_nm_cd8].copy()
adata_nm_cd8.obs['time_cond'] = (
    adata_nm_cd8.obs['time'].astype(str) + ' ' + adata_nm_cd8.obs['condition'].astype(str)
)
adata_nm_cd8 = adata_nm_cd8[:, ~adata_nm_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

print(f'CD8 PT/PB: {adata_pb_cd8.n_obs}')
print(adata_pb_cd8.obs['time_cond'].value_counts().to_string())
print(f'\nCD8 PT/NALM: {adata_nm_cd8.n_obs}')
print(adata_nm_cd8.obs['time_cond'].value_counts().to_string())

plot_marker_panel_violins(
    adata_pb_cd8, 'cd8_t_cell_markers',
    group_key='time_cond',
    adata_compare=adata_nm_cd8,
    primary_label='PT/PB',
    compare_label='PT/NALM',
)


## LFC scatter — Blinatumomab vs Mock at 6h

PT/NALM has no 48h Mock, so the LFC (Blina/Mock) comparison is restricted to **6h**.


In [ ]:
# [7 · LFC scatter — Blina vs Mock, PT/PB (x) vs PT/NALM (y), CD8, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_PT_PB, sys_b=SYS_PT_NALM,
    cell_type='CD8',
    label_a='PT/PB', label_b='PT/NALM',
    color_above='#9467bd',  # higher LFC in PT/NALM (above y=x)
    color_below='#2ca02c',  # higher LFC in PT/PB    (below y=x)
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()


## Mock vs Blinatumomab — abundance + spatial, per system


In [ ]:
# [8 · CD8 Blina vs Mock — abundance + spatial, per system, 6h]
# 12c-style 2x2 panel (abundance up/down, spatial coloc up/down) per system.
for sys_label, sys_val in [('PT/PB', SYS_PT_PB), ('PT/NALM', SYS_PT_NALM)]:
    plot_condition_comparison(
        adata,
        time_val='6h',
        cell_system=sys_val,
        cell_type='CD8',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )


## Spatial subsets for selected-marker comparisons


In [ ]:
# [9 · Spatial subsets — CD8 cells, per condition / system]
def _sp_subset(adata_full, time_val, cond_val, system_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'CD8') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    sp = sub.obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

print('Building spatial subsets (CD8 only):')
sp_6h_mock_pb   = _sp_subset(adata, '6h',  'Mock',         SYS_PT_PB)
sp_6h_mock_nm   = _sp_subset(adata, '6h',  'Mock',         SYS_PT_NALM)
sp_6h_blina_pb  = _sp_subset(adata, '6h',  'Blinatumomab', SYS_PT_PB)
sp_6h_blina_nm  = _sp_subset(adata, '6h',  'Blinatumomab', SYS_PT_NALM)
sp_48h_blina_pb = _sp_subset(adata, '48h', 'Blinatumomab', SYS_PT_PB)
sp_48h_blina_nm = _sp_subset(adata, '48h', 'Blinatumomab', SYS_PT_NALM)

all_sp_cols = sp_6h_mock_pb.columns


## CD8 immune synapse — PT/PB vs PT/NALM at 6h Blinatumomab


In [ ]:
# [12 · CD8 immune synapse: heatmaps + networks at 6h Blina]
SYNAPSE_CATEGORIES = {
    'cSMAC (signaling core)': (['CD3e', 'CD8', 'CD2', 'CD28', 'CD134', 'CD137',
                                'CD226', 'TIGIT', 'CD279', 'VISTA'],       '#e41a1c'),
    'pSMAC (adhesion ring)':  (['CD11a', 'CD50', 'KLRG1', 'CD94', 'CD48',
                                'CD352', 'CD53'],                           '#4daf4a'),
    'Exclusion zone':         (['CD45', 'CD43', 'CD44'],                    '#377eb8'),
}

mat_syn_pb, mat_syn_nm, mat_syn_diff = plot_synapse_suite(
    sp_a=sp_6h_blina_pb, sp_b=sp_6h_blina_nm,
    categories=SYNAPSE_CATEGORIES,
    label_a='PT/PB 6h Blina', label_b='PT/NALM 6h Blina',
    diff_label='Diff (PT/PB − PT/NALM)',
    suite_name='Immune synapse',
    highlight_node='CD3e',
    var_filter=adata.var_names,
    cluster_k=[3, 5],
)


# B cells — PT/PB vs PT/NALM

The B compartment differs across the two systems: **patient primary B-ALL cells**
(PT/PB) vs **NALM-6 cell line** (PT/NALM), under the shared patient T donor.
Differences here reflect the gap between primary patient leukemic biology and
the NALM-6 model rather than a donor effect.


## Cross-condition comparison — B cells across time × condition


In [ ]:
# [B-2 · 4-way DA panel: B cells — PT/PB vs PT/NALM across time × condition]
# (PT/NALM has no 48h Mock, so that group is missing on the comparison side.)
mask_pb_b = (
    (adata.obs['cell_system'] == SYS_PT_PB) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_pb_b = adata[mask_pb_b].copy()
adata_pb_b.obs['time_cond'] = (
    adata_pb_b.obs['time'].astype(str) + ' ' + adata_pb_b.obs['condition'].astype(str)
)

mask_nm_b = (
    (adata.obs['cell_system'] == SYS_PT_NALM) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_nm_b = adata[mask_nm_b].copy()
adata_nm_b.obs['time_cond'] = (
    adata_nm_b.obs['time'].astype(str) + ' ' + adata_nm_b.obs['condition'].astype(str)
)

print(f'B PT/PB: {adata_pb_b.n_obs}')
print(adata_pb_b.obs['time_cond'].value_counts().to_string())
print(f'\nB PT/NALM: {adata_nm_b.n_obs}')
print(adata_nm_b.obs['time_cond'].value_counts().to_string())

# Use the B-cell panel (currently keyed as 'or' in marker_panels.json).
plot_marker_panel_violins(
    adata_pb_b, 'or',
    group_key='time_cond',
    adata_compare=adata_nm_b,
    primary_label='PT/PB',
    compare_label='PT/NALM',
)


## LFC scatter — Blinatumomab vs Mock at 6h, B cells

PT/NALM has no 48h Mock, so the LFC (Blina/Mock) comparison is restricted to **6h**.


In [ ]:
# [B-7 · LFC scatter — Blina vs Mock, PT/PB (x) vs PT/NALM (y), B cells, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_PT_PB, sys_b=SYS_PT_NALM,
    cell_type='B',
    label_a='PT/PB', label_b='PT/NALM',
    color_above='#9467bd',  # higher LFC in PT/NALM (above y=x)
    color_below='#2ca02c',  # higher LFC in PT/PB    (below y=x)
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()


## Mock vs Blinatumomab — abundance + spatial, per system, B cells


In [ ]:
# [B-8 · B cell Blina vs Mock — abundance + spatial, per system, 6h]
# 12c-style 2x2 panel (abundance up/down, spatial coloc up/down) per system.
for sys_label, sys_val in [('PT/PB', SYS_PT_PB), ('PT/NALM', SYS_PT_NALM)]:
    plot_condition_comparison(
        adata,
        time_val='6h',
        cell_system=sys_val,
        cell_type='B',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )


## Spatial subsets — B cells, 6h Blina


In [ ]:
# [B-9 · Spatial subsets — B cells, 6h Blina PT/PB & PT/NALM]
def _sp_subset_b(adata_full, time_val, cond_val, system_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'B') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    sp = sub.obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

print('Building B-cell spatial subsets:')
sp_b_6h_blina_pb = _sp_subset_b(adata, '6h', 'Blinatumomab', SYS_PT_PB)
sp_b_6h_blina_nm = _sp_subset_b(adata, '6h', 'Blinatumomab', SYS_PT_NALM)

all_sp_cols_b = sp_b_6h_blina_pb.columns


## B cell APC synapse — PT/PB vs PT/NALM at 6h Blinatumomab

Antigen-presentation complex on the B-target side: MHC-II + costimulation +
inhibitory ligands + co-receptors + adhesion. Asks how primary patient B-ALL
organises its synapse-facing surface relative to the NALM-6 model under the
same patient T donor.


In [ ]:
# [B-12 · B cell APC synapse: heatmaps + networks at 6h Blina]
APC_SYNAPSE_CATEGORIES = {
    'MHC Class II (Ag presentation)':  (['HLA-DR-DP-DQ', 'HLA-DR', 'HLA-DQ', 'HLA-ABC'], '#e41a1c'),
    'Costimulation':                    (['CD80', 'CD86', 'CD40'],                        '#ff7f00'),
    'Inhibitory / Checkpoint ligands':  (['CD274', 'CD273', 'CD32', 'CD72', 'CD305'],     '#377eb8'),
    'B cell co-receptors':              (['CD19', 'CD20', 'CD22', 'CD79a'],               '#4daf4a'),
    'Adhesion (pSMAC)':                 (['CD54', 'CD58', 'CD50', 'CD102'],               '#984ea3'),
}

mat_apc_pb, mat_apc_nm, mat_apc_diff = plot_synapse_suite(
    sp_a=sp_b_6h_blina_pb, sp_b=sp_b_6h_blina_nm,
    categories=APC_SYNAPSE_CATEGORIES,
    label_a='PT/PB 6h Blina', label_b='PT/NALM 6h Blina',
    diff_label='Diff (PT/PB − PT/NALM)',
    suite_name='APC synapse',
    highlight_node='HLA-DR-DP-DQ',
    var_filter=adata.var_names,
    cluster_k=[3, 5],
)
